# COMP219
## Lab 7.2: CNN and Defence Strategies
The first part of Lab 7 provided a description of CNNs and some attack strategies were demonstrated. In the second part, we provide a description and implementation of some defence strategies. These strategies are designed to be universally applicable. For more options, please refer to the book Machine Learning Safety, which was made available as part of the module.

The first block of code implements the same CNN. The model is trained in the first cell; you may skip it if you have the model exported from part 1 to save you some time (in this case, you may jump to the second cell of code).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import argparse
import time
import os
from torch.autograd import Variable
from typing import Optional, Tuple
from tqdm import tqdm

model_dir = './defence_models'

print("CUDA available:", torch.cuda.is_available())

# Setup training parameters
parser = argparse.ArgumentParser(description='PyTorch MNIST Training')
parser.add_argument('--batch-size', type=int, default=128, metavar='N',
                    help='input batch size for training (default: 128)')
parser.add_argument('--test-batch-size', type=int, default=128, metavar='N',
                    help='input batch size for testing (default: 128)')
parser.add_argument('--epochs', type=int, default=20, metavar='N',
                    help='number of epochs to train')
parser.add_argument('--lr', type=float, default=0.01, metavar='LR',
                    help='learning rate')
parser.add_argument('--no-cuda', action='store_true', default=False,
                    help='disables CUDA training')
parser.add_argument('--seed', type=int, default=1, metavar='S',
                    help='random seed (default: 1)')
parser.add_argument('--model-dir', default='./model-mnist-cnn',
                    help='directory of model for saving checkpoint')
parser.add_argument('--load-model', action='store_true', default=False,
                    help='load model or not')

args = parser.parse_args(args=[]) 

if not os.path.exists(args.model_dir):
    os.makedirs(args.model_dir)
        
# Judge cuda is available or not
use_cuda = not args.no_cuda and torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print("Using device: ", device)

torch.manual_seed(args.seed)
kwargs = {'num_workers': 1, 'pin_memory': True} if use_cuda else {}

# Setup data loader
transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
        ])
trainset = datasets.MNIST('../data', train=True, download=True,
                   transform=transform)
testset = datasets.MNIST('../data', train=False,
                   transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=args.batch_size, shuffle=True, **kwargs)
test_loader = torch.utils.data.DataLoader(testset, batch_size=args.test_batch_size, shuffle=False, **kwargs)

# Define CNN
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output


# Train function with tqdm
def train(args, model, device, train_loader, optimizer, epoch):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)
    for batch_idx, (data, target) in enumerate(pbar):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        loss = F.cross_entropy(model(data), target)
        loss.backward()
        optimizer.step()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})


# Predict function with tqdm
def eval_test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    pbar = tqdm(test_loader, desc="Evaluating", leave=False)
    with torch.no_grad():
        for data, target in pbar:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item()
            pred = output.max(1, keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset)
    test_accuracy = correct / len(test_loader.dataset)
    return test_loss, test_accuracy


def evaluate_model(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    pbar = tqdm(data_loader, desc="Evaluating", leave=False)
    with torch.no_grad():
        for data, target in pbar:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    accuracy = correct / total
    return accuracy


CUDA available: True
Using device:  cuda


In [2]:
# Main function, train the initial model or load the model
def main():
    model = Net().to(device)
    optimizer = optim.SGD(model.parameters(), lr=args.lr)
    
    if args.load_model:
        # Load model
        model.load_state_dict(torch.load(os.path.join(args.model_dir, 'final_model.pt')))
        trnloss, trnacc = eval_test(model, device, train_loader)
        tstloss, tstacc = eval_test(model, device, test_loader)
        print('trn_loss: {:.4f}, trn_acc: {:.2f}%'.format(trnloss, 100. * trnacc), end=', ')
        print('test_loss: {:.4f}, test_acc: {:.2f}%'.format(tstloss, 100. * tstacc))
        
    else:
        # Train initial model
        for epoch in range(1, args.epochs + 1):
            start_time = time.time()

            #training
            train(args, model, device, train_loader, optimizer, epoch)

            #get trnloss and testloss
            trnloss, trnacc = eval_test(model, device, train_loader)
            tstloss, tstacc = eval_test(model, device, test_loader)

            #print trnloss and testloss
            print('Epoch '+str(epoch)+': '+str(int(time.time()-start_time))+'s', end=', ')
            print('trn_loss: {:.4f}, trn_acc: {:.2f}%'.format(trnloss, 100. * trnacc), end=', ')
            print('test_loss: {:.4f}, test_acc: {:.2f}%'.format(tstloss, 100. * tstacc))
        
        #save model
        torch.save(model.state_dict(), os.path.join(args.model_dir, 'final_model.pt'))

if __name__ == '__main__':
    main()


Epoch 1: 30s, trn_loss: 0.2896, trn_acc: 91.02%, test_loss: 0.2705, test_acc: 91.70%


Epoch 2: 30s, trn_loss: 0.1895, trn_acc: 94.33%, test_loss: 0.1752, test_acc: 94.74%


Epoch 3: 30s, trn_loss: 0.1373, trn_acc: 95.96%, test_loss: 0.1343, test_acc: 95.84%


Epoch 4: 30s, trn_loss: 0.1144, trn_acc: 96.60%, test_loss: 0.1143, test_acc: 96.38%


Epoch 5: 30s, trn_loss: 0.0948, trn_acc: 97.15%, test_loss: 0.0955, test_acc: 97.08%


...

Epoch 16: 31s, trn_loss: 0.0310, trn_acc: 99.13%, test_loss: 0.0524, test_acc: 98.35%


Epoch 17: 30s, trn_loss: 0.0371, trn_acc: 98.81%, test_loss: 0.0614, test_acc: 98.07%


Epoch 18: 31s, trn_loss: 0.0308, trn_acc: 99.08%, test_loss: 0.0548, test_acc: 98.24%


Epoch 19: 34s, trn_loss: 0.0238, trn_acc: 99.34%, test_loss: 0.0503, test_acc: 98.37%


Epoch 20: 33s, trn_loss: 0.0284, trn_acc: 99.12%, test_loss: 0.0572, test_acc: 98.12%


In [3]:
#setup training parameters
model = Net().to(device)
model.load_state_dict(torch.load(os.path.join(args.model_dir, 'final_model.pt')))

<All keys matched successfully>

### Defence Strategy: FGSM
Fast gradient sign method (FGSM) is a technique used in adversarial training which generates adversarial examples. It was introduced as a method to produce perturbed images that fool a neural network. The images are generated to maximise the model's error; FGSM performs this by applying modification to the original data based on the gradient of the loss function with respect to the input. In a defence scenario, this is integrated into the training process, which makes the model more resilient to perturbation-based attacks.

#### Why Use FGSM?
FGSM is cheaper and quicker compared to other adversarial training techniques we will demonstrate in this session. It takes one training iteration (forward-backward pass) to produce adversarial examples. FGSM is perhaps the most straightforward example of a defence strategy.

#### How Does FGSM Work?
During the training process, the model takes in a batch of input data and then produces the gradients of the loss function with respect to the input. However, the input data is perturbed by an amount which is determined by epsilon. The model is trained on the adversarial data.

In [4]:
def adversarial_training_fgsm(
    model: nn.Module,
    device: torch.device,
    train_loader,
    optimizer,
    epsilon: float = 0.3,
    epochs: int = 20,
    model_dir: str = './model-fgsm',
    save_model: bool = True,
    random_start: bool = False,
    mix_clean: bool = False,            # if True, train on 0.5*clean + 0.5*adv
    print_every: int = 100
) -> None:
    if save_model and not os.path.exists(model_dir):
        os.makedirs(model_dir)

    model.to(device)
    model.train()

    for epoch in range(1, epochs + 1):
        epoch_start_time = time.time()
        running_loss = 0.0
        running_adv_loss = 0.0
        running_clean_loss = 0.0
        total = 0

        loop = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False)

        for batch_idx, (data, target) in enumerate(loop, start=1):
            data = data.to(device)
            target = target.to(device)

            # random start (FGSM-RS)
            if random_start:
                delta = torch.empty_like(data).uniform_(-epsilon, epsilon).to(device)
                data_start = torch.clamp(data + delta, 0.0, 1.0).detach()
            else:
                data_start = data.detach()

            data_for_grad = data_start.clone().detach().requires_grad_(True)

            model.zero_grad()
            optimizer.zero_grad()

            output = model(data_for_grad)
            loss = F.cross_entropy(output, target)
            loss.backward()

            perturbation = epsilon * data_for_grad.grad.data.sign()
            adv_data = torch.clamp(data_start + perturbation, 0.0, 1.0).detach()

            model.zero_grad()
            optimizer.zero_grad()

            output_adv = model(adv_data)
            loss_adv = F.cross_entropy(output_adv, target)

            if mix_clean:
                output_clean = model(data)
                loss_clean = F.cross_entropy(output_clean, target)
                loss_total = 0.5 * loss_clean + 0.5 * loss_adv
                running_clean_loss += loss_clean.item() * data.size(0)
            else:
                loss_total = loss_adv
                running_clean_loss += 0.0

            loss_total.backward()
            optimizer.step()

            running_adv_loss += loss_adv.item() * data.size(0)
            running_loss += loss_total.item() * data.size(0)
            total += data.size(0)

            if (batch_idx % print_every) == 0:
                avg_loss = running_loss / total
                avg_adv = running_adv_loss / total
                loop.set_postfix(avg_loss=f"{avg_loss:.4f}", adv_loss=f"{avg_adv:.4f}")

        epoch_time = int(time.time() - epoch_start_time)
        print(f"Epoch {epoch} completed in {epoch_time}s — avg_loss={running_loss/total:.4f}, avg_adv_loss={running_adv_loss/total:.4f}")

        if save_model:
            model_path = os.path.join(model_dir, f'fgsm_def_epoch{epoch}.pt')
            torch.save(model.state_dict(), model_path)

    if save_model:
        model_path = os.path.join(model_dir, 'fgsm_def_final.pt')
        torch.save(model.state_dict(), model_path)
        print(f"Model saved to {model_path}")


def evaluate_adversarial_training_fgsm(
    model: nn.Module,
    device: torch.device,
    test_loader,
    epsilon: float = 0.3
) -> Tuple[float, float]:
    model.to(device)
    model.eval()
    correct_clean = 0
    correct_adv = 0
    total = 0

    loop = tqdm(test_loader, desc='Evaluating', leave=False)

    with torch.no_grad():
        for data, target in loop:
            data = data.to(device)
            target = target.to(device)
            total += target.size(0)

            out_clean = model(data)
            pred_clean = out_clean.argmax(dim=1)
            correct_clean += (pred_clean == target).sum().item()

    # adversarial evaluation needs gradient
    loop_adv = tqdm(test_loader, desc='Evaluating Adv', leave=False)
    for data, target in loop_adv:
        data = data.to(device)
        target = target.to(device)

        data_req = data.clone().detach().requires_grad_(True)
        model.zero_grad()

        out = model(data_req)
        loss = F.cross_entropy(out, target)
        loss.backward()
        perturbation = epsilon * data_req.grad.data.sign()
        adv_data = torch.clamp(data + perturbation, 0.0, 1.0)

        with torch.no_grad():
            out_adv = model(adv_data)
            pred_adv = out_adv.argmax(dim=1)
            correct_adv += (pred_adv == target).sum().item()

    clean_accuracy = correct_clean / total
    adv_accuracy = correct_adv / total
    return clean_accuracy, adv_accuracy

In [5]:
#device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", device)

#data loading for MNIST dataset
print("load dataset")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
trainset = datasets.MNIST('../data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

print("initialise model")
#initialise model, optimiser, and training parameters
model = Net().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)

print("adversarial training")

#call FGSM-based adversarial training function
adversarial_training_fgsm(model, device, train_loader, optimizer, epsilon=0.3, epochs=5)
clean_acc, adv_acc = evaluate_adversarial_training_fgsm(model, device, test_loader, epsilon=0.3)

#evaluate adversarially trained data
print("\nEvaluation Results for FGSM:")
print(f"Accuracy on clean test data: {clean_acc * 100:.2f}%")
print(f"Accuracy on adversarial (FGSM) test data: {adv_acc * 100:.2f}%")
print(f"Drop in accuracy due to adversarial attack: {clean_acc * 100 - adv_acc * 100:.2f}%")

Using device:  cuda
load dataset
initialise model
adversarial training


Epoch 1 completed in 16s — avg_loss=1.0219, avg_adv_loss=1.0219


Epoch 2 completed in 15s — avg_loss=0.4394, avg_adv_loss=0.4394


Epoch 3 completed in 17s — avg_loss=0.3354, avg_adv_loss=0.3354


Epoch 4 completed in 16s — avg_loss=0.2705, avg_adv_loss=0.2705


Epoch 5 completed in 16s — avg_loss=0.2319, avg_adv_loss=0.2319
Model saved to ./model-fgsm\fgsm_def_final.pt



Evaluation Results for FGSM:
Accuracy on clean test data: 96.22%
Accuracy on adversarial (FGSM) test data: 93.44%
Drop in accuracy due to adversarial attack: 2.78%


### Defence Strategy: PGD
Projected gradient descent (PGD) is a common method for generating adversarial examples by introducing perturbations to input images. PGD iteratively adjusts each input image to maximise the loss function (remember ideally, we want to minimise this value). PGD is considered to be a benchmark for testing adversarial robustness since it provides strong baseline for creating adversarially perturbed data.

In adversarial training, PGD-generated adversarial samples are incorporated directly into training to improve the robustness of the model.

#### Why Use PGD?
PGD is effective as it can simulate an iterative adversarial attack that works well across many models and datasets. It iterates multiple times in order to refine the added perturbation to the data. Using this method in adversarial training makes the model more robust to a range of adversarial inputs by training directly on adversarial examples.

Downside: it takes a long time to train the model.

#### How Does PGD Work?
PGD is similar to FGSM, the difference is that a projection step is added.
1. Initialisation: for each input, a copy is made to create an adversarial version of the training data. The copy is modified but it is kept similar to the original image (determined by epsilon).
2. Gradient Descent with Perturbation: with each iteration, the model calculates the gradient of the loss with respect to the input data. A small step (defined by alpha) is added in the direction of the gradient's sign to increase loss. This leads to the adversarial example being pushed further away from the correct classification boundary - this simulates a worst-case perturbation based on current model weights.
3. Projection: after each updated, the perturbed example is projected back within an acceptable range around the original input (in other words, the values are clamped). This forces the perturbed data to stay within the epsilon constraint.
4. Iteration: repeat steps 2 and 3 for a number of iterations.

In adversarial training, the PGD-based adversarial examples are used directly in the training loop to update the model weights. The process is computationally intensive(!) but is highly effective. 

In [6]:
def adversarial_training_pgd(
    model: nn.Module,
    device: torch.device,
    train_loader,
    optimizer,
    epsilon: float = 0.3,
    alpha: float = 0.01,
    num_iter: int = 40,
    epochs: int = 20,
    model_dir: str = './model-pgd',
    save_model: bool = True,
    random_start: bool = True
):
    if save_model and not os.path.exists(model_dir):
        os.makedirs(model_dir)
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_start_time = time.time()
        loop = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False)
        
        for batch_idx, (data, target) in enumerate(loop):
            data, target = data.to(device), target.to(device)
            
            # Initialize adversarial data
            if random_start:
                adv_data = data + torch.empty_like(data).uniform_(-epsilon, epsilon)
                adv_data = torch.clamp(adv_data, 0, 1)
            else:
                adv_data = data.clone().detach()
            
            adv_data.requires_grad_(True)
            
            # PGD iterations
            for _ in range(num_iter):
                optimizer.zero_grad()
                if adv_data.grad is not None:
                    adv_data.grad.zero_()
                
                output = model(adv_data)
                loss = F.cross_entropy(output, target)
                loss.backward()
                
                # Update adversarial data
                adv_data.data = adv_data.data + alpha * adv_data.grad.sign()
                adv_data.data = torch.max(torch.min(adv_data.data, data + epsilon), data - epsilon)
                adv_data.data = torch.clamp(adv_data.data, 0, 1)
            
            # Update model on adversarial data
            optimizer.zero_grad()
            output_adv = model(adv_data)
            loss_adv = F.cross_entropy(output_adv, target)
            loss_adv.backward()
            optimizer.step()
        
        print(f"Epoch {epoch} completed in {int(time.time() - epoch_start_time)}s")

    if save_model:
        model_path = os.path.join(model_dir, 'pgd_att.pt')
        torch.save(model.state_dict(), model_path)
        print(f"Model saved to {model_path}")


def evaluate_adversarial_training_pgd(
    model: nn.Module,
    device: torch.device,
    test_loader,
    epsilon: float = 0.3,
    alpha: float = 0.01,
    num_iter: int = 40,
    random_start: bool = True
):
    model.eval()
    correct_clean = 0
    correct_adv = 0
    total = 0

    loop = tqdm(test_loader, desc='Evaluating', leave=False)

    for data, target in loop:
        data, target = data.to(device), target.to(device)

        # Clean accuracy
        with torch.no_grad():
            output_clean = model(data)
            pred_clean = output_clean.argmax(dim=1)
            correct_clean += (pred_clean == target).sum().item()

        # Adversarial PGD attack
        if random_start:
            adv_data = data + torch.empty_like(data).uniform_(-epsilon, epsilon)
            adv_data = torch.clamp(adv_data, 0, 1)
        else:
            adv_data = data.clone().detach()
        adv_data.requires_grad_(True)

        for _ in range(num_iter):
            if adv_data.grad is not None:
                adv_data.grad.zero_()
            output = model(adv_data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            adv_data.data = adv_data.data + alpha * adv_data.grad.sign()
            adv_data.data = torch.max(torch.min(adv_data.data, data + epsilon), data - epsilon)
            adv_data.data = torch.clamp(adv_data.data, 0, 1)

        # Adversarial accuracy
        output_adv = model(adv_data)
        pred_adv = output_adv.argmax(dim=1)
        correct_adv += (pred_adv == target).sum().item()
        total += target.size(0)

    clean_accuracy = correct_clean / total
    adv_accuracy = correct_adv / total
    return clean_accuracy, adv_accuracy


In [7]:
#device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", device)

#data loading for MNIST dataset
print("load dataset")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
trainset = datasets.MNIST('../data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
print("initialise model")

#initalise model, optimiser, and training parameters
model = Net().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)

print("adversarial training")

#call adversarial training function
adversarial_training_pgd(model, device, train_loader, optimizer, epsilon=0.3, alpha=0.01, num_iter=5, epochs=5)
clean_accuracy, adv_accuracy = evaluate_adversarial_training_pgd(model, device, test_loader, epsilon=0.3, alpha=0.01, num_iter=40)

print("\nEvaluation Results for PGD:")
print(f"Accuracy on clean test data: {clean_accuracy * 100:.2f}%")
print(f"Accuracy on adversarial (PGD) test data: {adv_accuracy * 100:.2f}%")
print(f"Drop in accuracy due to adversarial attack: {clean_accuracy * 100 - adv_accuracy * 100:.2f}%")

Using device:  cuda
load dataset
initialise model
adversarial training


Epoch 1 completed in 14s


Epoch 2 completed in 14s


Epoch 3 completed in 13s


Epoch 4 completed in 13s


Epoch 5 completed in 14s
Model saved to ./model-pgd\pgd_att.pt



Evaluation Results for PGD:
Accuracy on clean test data: 94.84%
Accuracy on adversarial (PGD) test data: 90.64%
Drop in accuracy due to adversarial attack: 4.20%


### Defence Strategy: MMAT
Min-Max Adversarial Training (MMAT) aims to train the model to handle worst-case adversarial perturbations. This algorithm generates adversarial examples by optimising a maximisation step that increases the model's loss, followed by a minimisation step where the model adjusts to minimise this (maximised) loss. Like PGD, this method is iterative.

#### Why Use MMAT?
MMAT prepares the model to handle the most challenging adversarial perturbation-based attacks. Like previous examples, this strategy trains the model on adversarial examples.

#### How Does MMAT Work?
The algorithm works by combining the following two steps:
1. Maximisation step: in each iteration, an adversarial example is generated by finding the perturbation direction that maximises the model's loss within a predefined limit (epsilon). This is done by adjusting the input in the gradient direction iteratively over multiple steps.
2. Minimisation step: after generating the adversarial example, the model is trained on this perturbed data by minimising the loss function on these adversarial inputs. This step updates the model's parameters to adapt to the adversarial examples.

In [8]:
# MMAT (Moderate‑Margin Adversarial Training) pseudocode
# Purpose: train a model to be robust by generating per-sample, moderate adversarial examples
# (not always worst-case) and using them in the outer minimisation (model update).

function MMAT_Train(model, train_loader, optimizer,
                    epochs,
                    # inner-attack / margin estimation hyperparams
                    inner_steps = 10, inner_step_size = 0.01, inner_epsilon_max = 0.3,
                    # moderate sampling / mixing policy
                    sample_fraction = 0.5,    # fraction of samples to replace with adversarials each batch
                    margin_scale = 0.7,       # scale of margin to use for "moderate" perturbation (0..1)
                    pgd_restarts = 1,
                    # training control
                    device = 'cpu'):

    # model: network to train
    # train_loader: yields batches of (x_batch, y_batch) in the model's expected normalised space
    # inner_steps/inner_step_size: used for inner (PGD-like) maximisation when estimating margins or crafting adversarials
    # inner_epsilon_max: maximum allowed perturbation used for margin search
    # sample_fraction: how many examples per batch are replaced by moderate adversarials
    # margin_scale: fraction of the estimated per-sample margin used to craft "moderate" examples
    # pgd_restarts: optional restarts for inner maximiser (for more reliable margin estimates)

    for epoch in 1..epochs:
        for (x_batch, y_batch) in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            # ---- Step A: (Optional) quick per-sample margin estimation ----
            # Purpose: estimate how far each sample is from the decision boundary,
            # so we can craft a moderate perturbation instead of always using the worst-case epsilon.
            # We use a cheap approximate search: short PGD with increasing eps or a small binary search.
            margins = zeros(len(x_batch))   # will store estimated distance-to-boundary (in same norm as inner attack)
            for i in 0 .. len(x_batch)-1:
                x = x_batch[i:i+1].clone()
                y = y_batch[i:i+1]
                # simple margin estimate: try PGD with a few increasing eps steps until misclassified
                # start small and grow eps until model misclassifies or hit inner_epsilon_max
                found = False
                eps_try = inner_step_size
                while eps_try <= inner_epsilon_max and not found:
                    x_adv = PGD_Attack(model, x, y, eps=eps_try, steps=inner_steps, step_size=inner_step_size, restarts=pgd_restarts)
                    pred = argmax(model(x_adv))
                    if pred != y:
                        margins[i] = eps_try
                        found = True
                    else:
                        eps_try = eps_try * 1.5   # grow candidate epsilon (coarse search)
                if not found:
                    margins[i] = inner_epsilon_max   # treat as "hard to flip" (use max)

            # ---- Step B: Generate moderate adversarial examples ----
            # For a subset of samples (controlled by sample_fraction), craft adversarials at a
            # scaled fraction of the estimated margin (so they are "moderate", not necessarily worst-case).
            batch_size = len(x_batch)
            num_to_replace = int(sample_fraction * batch_size)
            indices_to_replace = SELECT_INDICES_TO_REPLACE(margins, num_to_replace)
            # SELECT_INDICES_TO_REPLACE could be random, or prioritise small margins, or hardest examples

            x_mixed = x_batch.clone()
            for idx in indices_to_replace:
                eps_use = margin_scale * margins[idx]     # scale per-sample margin to get "moderate" eps
                # craft adversarial at eps_use (PGD or other inner maximiser)
                x_adv = PGD_Attack(model, x_batch[idx:idx+1], y_batch[idx:idx+1],
                                   eps=eps_use, steps=inner_steps, step_size=inner_step_size, restarts=pgd_restarts)
                x_mixed[idx:idx+1] = x_adv.detach()

            # ---- Step C: Outer minimisation — update model parameters on mixed batch ----
            optimizer.zero_grad()
            logits = model(x_mixed)
            loss = CrossEntropyLoss(logits, y_batch)
            loss.backward()
            optimizer.step()

        # optional: evaluate on clean validation set and on strong adversarial set
        # optional: adjust sample_fraction or margin_scale over epochs (curriculum)

    return model


# Helper: a typical PGD inner maximiser (L_inf norm example)
function PGD_Attack(model, x, y, eps, steps, step_size, restarts=1):
    best_adv = x.clone()
    best_loss = -inf
    for r in 1..restarts:
        # random start inside L_inf ball
        x_adv = x + uniform(-eps, eps)
        x_adv = clamp(x_adv, data_min, data_max)
        for t in 1..steps:
            x_adv.requires_grad_(True)
            logits = model(x_adv)
            loss = CrossEntropyLoss(logits, y)   # attacker maximises this
            grad = gradient(loss, x_adv)
            x_adv = x_adv + step_size * sign(grad)
            # project back to L_inf ball and valid pixel range
            x_adv = clip(x_adv, x - eps, x + eps)
            x_adv = clamp(x_adv, data_min, data_max)
        # select highest-loss restart
        final_loss = CrossEntropyLoss(model(x_adv.detach()), y).item()
        if final_loss > best_loss:
            best_loss = final_loss
            best_adv = x_adv.detach().clone()
    return best_adv


SyntaxError: invalid syntax (3994861421.py, line 5)

### Closing Thoughts
In Part 1 of Lab 7, we demonstrated some attack strategies each of which generate adversarial data. Using the adversarial data, we attempted to fool the CNN and force it to make wrong predictions.

In Part 2, we introduced some defence strategies. Each of these strategies are implemented at the point of model training as our aim was to prepare the model for scenarios where some adversarial training data would be fed to it.

If you perform an attack on an adversarially trained model, you should observe:
- reduced impact of perturbation effect
- attacks take longer to succeed
- adversarially trained models have lower accuracy on the clean data than the standard model.

There are more techniques you can discover in the book Machine Learning Safety. Chapter 10 provides overviews of several deep learning models (see 10.4 for CNNs). You can find more strategies in chapters 10.7-10.11